In [10]:
import time

def parse_sam_file(sam_file):
    """Parse SAM file and return dictionary of chromosome to list of start positions. """
    chrom_to_starts = {}
    with open(sam_file, 'r') as f:
        for line in f:
            if line.startswith('@'):  # Skip header lines
                continue
            fields = line.strip().split('\t')
            if len(fields) < 4:
                continue
            chrom = fields[2]
            if chrom == '*':  # Unmapped read
                continue
            start = int(fields[3])
            if chrom not in chrom_to_starts:
                chrom_to_starts[chrom] = []
            chrom_to_starts[chrom].append(start)
    return chrom_to_starts

def parse_gtf_file(gtf_file):
    """Parse GTF file and return dictionary of gene info (exon coordinates)"""
    gene_info = {}
    with open(gtf_file, 'r') as f:
        for line in f:
            if line.startswith('#'):  # Skip comment lines
                continue
            fields = line.strip().split('\t')
            if len(fields) < 9:
                continue
            feature_type = fields[2]
            if feature_type != 'exon':  # We only care about exons
                continue
            chrom = fields[0]
            start = int(fields[3])
            end = int(fields[4])
            
            # Parse gene name from attributes
            attributes = fields[8]
            gene_name = None
            for attr in attributes.split(';'):
                attr = attr.strip()
                if attr.startswith('gene_name'):
                    gene_name = attr.split('"')[1]
                    break
            
            if gene_name:
                if gene_name not in gene_info:
                    gene_info[gene_name] = {'chrom': chrom, 'exons': []}
                gene_info[gene_name]['exons'].append((start, end))
    return gene_info

def count_matches_unsorted(chrom_to_starts, gene_info):
    """Count matches without sorting SAM file coordinates"""
    gene_counts = {}
    
    for gene_name, info in gene_info.items():
        chrom = info['chrom']
        if chrom not in chrom_to_starts:
            gene_counts[gene_name] = 0
            continue
        
        count = 0
        starts = chrom_to_starts[chrom]
        
        for exon_start, exon_end in info['exons']:
            for start in starts:
                if exon_start <= start <= exon_end:
                    count += 1
        
        gene_counts[gene_name] = count
    
    return gene_counts

def count_matches_sorted(chrom_to_starts, gene_info):
    """Count matches with sorted SAM file coordinates"""
    # Sort the start positions for each chromosome
    sorted_chrom_to_starts = {chrom: sorted(starts) for chrom, starts in chrom_to_starts.items()}
    
    gene_counts = {}
    
    for gene_name, info in gene_info.items():
        chrom = info['chrom']
        if chrom not in sorted_chrom_to_starts:
            gene_counts[gene_name] = 0
            continue
        
        count = 0
        starts = sorted_chrom_to_starts[chrom]
        starts_idx = 0  # Current position in the starts list
        starts_len = len(starts)
        
        for exon_start, exon_end in info['exons']:
            # Find the first start position >= exon_start
            while starts_idx < starts_len and starts[starts_idx] < exon_start:
                starts_idx += 1
            
            # Count all positions <= exon_end
            temp_idx = starts_idx
            while temp_idx < starts_len and starts[temp_idx] <= exon_end:
                count += 1
                temp_idx += 1
        
        gene_counts[gene_name] = count
    
    return gene_counts

def main():
    # Parse files
    sam_file = 'sample.sam'
    gtf_file = 'Arabidopsis.gtf'
    
    print("Parsing SAM file...")
    chrom_to_starts = parse_sam_file(sam_file)
    print("Parsing GTF file...")
    gene_info = parse_gtf_file(gtf_file)
    
    # Unsorted approach
    print("\nRunning unsorted approach...")
    start = time.process_time()
    unsorted_counts = count_matches_unsorted(chrom_to_starts, gene_info)
    unsorted_time = time.process_time() - start
    print(f"Unsorted approach took {unsorted_time:.2f} seconds")
    
    # Sorted approach
    print("\nRunning sorted approach...")
    start = time.process_time()
    sorted_counts = count_matches_sorted(chrom_to_starts, gene_info)
    sorted_time = time.process_time() - start
    print(f"Sorted approach took {sorted_time:.2f} seconds")
    
    # Verify counts match (should be True)
    print("\nVerifying counts match between approaches...")
    print(f"Counts match: {unsorted_counts == sorted_counts}")
    
    # Print results
    print("\nSample gene counts (first 5 genes):")
    for i, (gene, count) in enumerate(sorted_counts.items()):
        if i >= 5:
            break
        print(f"{gene}: {count} reads")
    
    # Time comparison explanation
    print("\nTime Comparison Explanation:")
    print("1. Unsorted Approach: O(n*m) complexity where n is number of reads and m is number of exons.")
    print("   - Must check every read against every exon")
    print("2. Sorted Approach: O(n log n) for sorting + O(n + m) for counting.")
    print("   - After sorting, we can efficiently skip ranges of reads that don't match")
    print("   - Typically much faster for large datasets as we avoid nested loops")

if __name__ == '__main__':
    main()

Parsing SAM file...
Parsing GTF file...

Running unsorted approach...
Unsorted approach took 323.59 seconds

Running sorted approach...
Sorted approach took 89.67 seconds

Verifying counts match between approaches...
Counts match: False

Sample gene counts (first 5 genes):
AT1G01010: 2 reads
AT1G01020: 2 reads
AT1G01030: 0 reads
AT1G01040: 7 reads
AT1G01046: 0 reads

Time Comparison Explanation:
1. Unsorted Approach: O(n*m) complexity where n is number of reads and m is number of exons.
   - Must check every read against every exon
2. Sorted Approach: O(n log n) for sorting + O(n + m) for counting.
   - After sorting, we can efficiently skip ranges of reads that don't match
   - Typically much faster for large datasets as we avoid nested loops
